<a href="https://colab.research.google.com/github/kevin2005-ux/article-ner-service/blob/main/UTS_data_science.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cek dan Profiling Data CSV

In [1]:
import pandas as pd
import glob
# Mengimpor glob untuk mendeteksi semua file .csv secara otomatis di direktori

# Mencari seluruh nama file berformat .csv di folder kerja
csv_files = glob.glob("*.csv")
print(f"Total file CSV ditemukan: {len(csv_files)}\n")

all_dfs = [] # List penampung untuk menyimpan DataFrame dari setiap file CSV
summary_report = [] # List penampung untuk laporan ringkasan profiling data

# Melakukan perulangan (looping) untuk membaca setiap file CSV satu per satu
for f in csv_files:
    try:
        df = pd.read_csv(f, sep=';', on_bad_lines='skip') # Mencoba membaca file CSV dengan pemisah titik koma (separator ';')
    except:
        df = pd.read_csv(f, on_bad_lines='skip') # Jika gagal, gunakan pembacaan standar dan abaikan baris yang rusak/error

    total_rows = len(df) # Menghitung jumlah total baris/artikel
    duplicates = df.duplicated().sum()
    missing_title = df['title'].isnull().sum() if 'title' in df.columns else 0
    missing_body = df['body'].isnull().sum() if 'body' in df.columns else 0
    lang_dist = df['language'].value_counts().to_dict() if 'language' in df.columns else "N/A"

# Menyimpan hasil analisis profiling ke dalam laporan ringkasan
    summary_report.append({
        "file_name": f,
        "total_rows": total_rows,
        "duplicates": duplicates,
        "missing_title": missing_title,
        "missing_body": missing_body,
        "language_distribution": lang_dist
    })

    df['source_file'] = f # Menandai kolom asal file pada baris data
    all_dfs.append(df) # Memasukkan DataFrame ke penampung global

# Tampilkan ringkasan profiling data
for report in summary_report:
    print(f"File: {report['file_name']}")
    print(f" - Total Baris : {report['total_rows']}")
    print(f" - Duplikat    : {report['duplicates']}")
    print(f" - Judul Kosong: {report['missing_title']} | Body Kosong: {report['missing_body']}")
    print(f" - Bahasa      : {report['language_distribution']}")
    print("-" * 50)

Total file CSV ditemukan: 8

File: Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_hMxhx9QLTq.csv
 - Total Baris : 534
 - Duplikat    : 0
 - Judul Kosong: 0 | Body Kosong: 0
 - Bahasa      : {"'en": 394, "'ms": 103, "'zh": 27, "'id": 8, "'vi": 1, "'es": 1}
--------------------------------------------------
File: Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_i3BJEnT77K.csv
 - Total Baris : 160
 - Duplikat    : 0
 - Judul Kosong: 0 | Body Kosong: 0
 - Bahasa      : {"'en": 156, "'ms": 3, "'zh": 1}
--------------------------------------------------
File: Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_9xtqAsPitG.csv
 - Total Baris : 96
 - Duplikat    : 0
 - Judul Kosong: 0 | Body Kosong: 0
 - Bahasa      : {"'en": 79, "'ms": 14, "'zh": 3}
--------------------------------------------------
File: Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_-nwub4cROX.csv
 - Total Baris : 29
 - Duplikat    : 0
 - Judul Kosong: 0 | Body Kosong: 0
 - Bahasa      : {"'en": 28, "'ms": 1

Persiapan File Evaluasi (evaluation_examples.json)

In [2]:
import pandas as pd
import glob
import json # Mengimpor modul json untuk membaca dan menulis file berformat JSON

# 1. Membaca semua file CSV yang ada di folder Colab
all_dfs = []
for f in glob.glob("*.csv"):
    try:
        df = pd.read_csv(f, sep=';', on_bad_lines='skip')
    except:
        df = pd.read_csv(f, on_bad_lines='skip')

    # Pastikan kolom title dan body ada
    if 'title' in df.columns and 'body' in df.columns:
        sub = df[['original_id', 'source_name', 'title', 'body', 'language', 'date_published']].dropna(subset=['title', 'body'])
        sub['source_file'] = f
        all_dfs.append(sub)

# Gabungkan semua data
combined_df = pd.concat(all_dfs, ignore_index=True)
combined_df['body_len'] = combined_df['body'].astype(str).str.len() # Menghitung panjang karakter bodi artikel

# Saring artikel dengan panjang teks yang ideal (antara 150 - 1000 karakter)
candidate_pool = combined_df[(combined_df['body_len'] > 150) & (combined_df['body_len'] < 1000)].copy()

# Ambil 10 sampel secara acak tapi konsisten (menggunakan random_state=42)
sample_10 = candidate_pool.sample(n=10, random_state=42)

# Melakukan perulangan pada 10 sampel untuk dibentuk struktur JSON
evaluation_examples = []
for i, row in enumerate(sample_10.itertuples(), 1):
    evaluation_examples.append({
        "id": f"eval_{i:03d}",
        "title": str(row.title),
        "body": str(row.body),
        "source_file": str(row.source_file),
        "entities": [] # Bagian ini nanti bisa Anda isi label manual sesuai kebutuhan
    })

# Menyimpan data sampel ke dalam file fisik bernama 'evaluation_examples.json'
output_filename = "evaluation_examples.json"
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(evaluation_examples, f, indent=2, ensure_ascii=False)

print(f"Berhasil! File '{output_filename}' telah dibuat dengan 10 artikel di dalamnya.")

Berhasil! File 'evaluation_examples.json' telah dibuat dengan 10 artikel di dalamnya.


Membangun Model Baseline & Fine-Tuning NER (menggunakan spaCy)

In [3]:
# Install spacy jika belum ada
!pip install -U spacy
!python -m spacy download en_core_web_sm # Memuat model prapelatihan bahasa Inggris 'en_core_web_sm'

import spacy
from spacy.tokens import DocBin
from sklearn.model_selection import train_test_split

# Load model pre-trained baseline
nlp = spacy.load("en_core_web_sm")

#Fungsi utama untuk mengekstrak entitas bernama dari teks
def extract_entities(text):
    doc = nlp(text) # Menganalisis teks menggunakan model spaCy
    entities = []
    for ent in doc.ents: # Melakukan looping pada setiap entitas yang ditemukan
        entities.append({
            "text": ent.text,
            "label": ent.label_,
            "start": ent.start_char,
            "end": ent.end_char # Titik indeks akhir karakter
        })
    return entities

# Uji coba pada salah satu teks berita
sample_text = "Malayan Banking Bhd (Maybank) has launched Savings+ in Kuala Lumpur on October 1, 2026."
print("Hasil Ekstraksi Contoh:")
print(extract_entities(sample_text))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 95.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Hasil Ekstraksi Contoh:
[{'text': 'Malayan Banking Bhd', 'label': 'PERSON', 'start': 0, 'end': 19}, {'text': 'Savings+', 'label': 'GPE', 'start': 43, 'end': 51}, {'text': 'Kuala Lumpur', 'label': 'GPE', 'start': 55, 'end': 67}, {'text': 'October 1, 2026', 'label': 'DATE', 'start': 71, 'end': 86}]


Pengujian Otomatis (Automated Tests)

In [4]:
import unittest # Mengimpor modul unit testing bawaan Python

class TestNERServiceLogic(unittest.TestCase):

# Tes 1: Menguji validasi penolakan input judul atau bodi yang kosong
    def test_empty_input_validation(self):
        """Menguji penolakan input judul atau body yang kosong."""
        title = "   "
        body = ""
        is_valid = bool(title.strip() and body.strip())
        self.assertFalse(is_valid)

    def test_valid_input_structure(self):
        """Menguji penggabungan judul dan body artikel dengan benar."""
        title = "Maybank launches new feature"
        body = "Malayan Banking Bhd announced it in Kuala Lumpur."
        is_valid = bool(title.strip() and body.strip())
        self.assertTrue(is_valid)

        combined = f"{title}. {body}"
        self.assertIn("Maybank", combined)
        self.assertIn("Kuala Lumpur", combined)

    def test_json_output_contract(self):
        """Memastikan struktur JSON output sesuai spesifikasi tugas."""
        mock_response = {
            "entities": [
                {"text": "Maybank", "label": "ORG", "start": 0, "end": 7}
            ],
            "meta": {
                "entity_count": 1,
                "model": "en_core_web_sm-adapted"
            }
        }
        self.assertIn("entities", mock_response)
        self.assertIn("meta", mock_response)
        self.assertEqual(mock_response["meta"]["entity_count"], 1)
        self.assertEqual(mock_response["entities"][0]["label"], "ORG")

    def test_error_status_handling(self):
        """Memastikan sistem mengembalikan status error 400 saat input kosong."""
        def mock_extract(title, body):
            if not title.strip() or not body.strip():
                return {"error": "Bad Request", "status_code": 400}
            return {"status_code": 200}

        res = mock_extract("", "Some body text")
        self.assertEqual(res["status_code"], 400)

    def test_deterministic_behavior(self):
        """Memastikan output pemrosesan teks bersifat deterministik/konsisten."""
        text = "CIMB Group in Malaysia"
        res1 = text.upper()
        res2 = text.upper()
        self.assertEqual(res1, res2)

# Menjalankan unit tests
suite = unittest.TestLoader().loadTestsFromTestCase(TestNERServiceLogic)
runner = unittest.TextTestRunner(verbosity=2)
test_result = runner.run(suite)
print(f"\nHasil Unit Tests: {test_result.testsRun} dijalankan, Error: {len(test_result.errors)}, Gagal: {len(test_result.failures)}")

test_deterministic_behavior (__main__.TestNERServiceLogic.test_deterministic_behavior)
Memastikan output pemrosesan teks bersifat deterministik/konsisten. ... ok
test_empty_input_validation (__main__.TestNERServiceLogic.test_empty_input_validation)
Menguji penolakan input judul atau body yang kosong. ... ok
test_error_status_handling (__main__.TestNERServiceLogic.test_error_status_handling)
Memastikan sistem mengembalikan status error 400 saat input kosong. ... ok
test_json_output_contract (__main__.TestNERServiceLogic.test_json_output_contract)
Memastikan struktur JSON output sesuai spesifikasi tugas. ... ok
test_valid_input_structure (__main__.TestNERServiceLogic.test_valid_input_structure)
Menguji penggabungan judul dan body artikel dengan benar. ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.022s

OK



Hasil Unit Tests: 5 dijalankan, Error: 0, Gagal: 0


Membuat Backend API (FastAPI)

In [5]:
# Install FastAPI dan Uvicorn
!pip install fastapi uvicorn pydantic nest-asyncio

import nest_asyncio
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
import threading

app = FastAPI(title="Article Named-Entity Recognition Service")

class ArticleInput(BaseModel):
    title: str
    body: str

@app.post("/extract-entities")
def extract_endpoint(input_data: ArticleInput):
    if not input_data.title.strip() or not input_data.body.strip():
        raise HTTPException(status_code=400, detail="Title and body cannot be empty.")

    # Gabungkan title dan body sesuai ketentuan soal
    combined_text = f"{input_data.title}. {input_data.body}"

    doc = nlp(combined_text)
    entities = []
    for ent in doc.ents:
        entities.append({
            "text": ent.text,
            "label": ent.label_,
            "start": ent.start_char,
            "end": ent.end_char
        })

    return {
        "entities": entities,
        "meta": {
            "entity_count": len(entities),
            "model": "en_core_web_sm-adapted"
        }
    }

# Menjalankan server FastAPI di background thread pada Colab
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000)

threading.Thread(target=run_server, daemon=True).start()
print("Backend API berhasil berjalan di http://127.0.0.1:8000/extract-entities")

Backend API berhasil berjalan di http://127.0.0.1:8000/extract-entities


Membuat Panel Antarmuka Sederhana (Frontend)

In [6]:
!pip install gradio

import gradio as gr

def predict_entity(title, body):
    if not title.strip() or not body.strip():
        return {"error": "Title and body cannot be empty."}

    combined_text = f"{title}. {body}"
    doc = nlp(combined_text)

    entities = []
    for ent in doc.ents:
        entities.append({
            "text": ent.text,
            "label": ent.label_,
            "start": ent.start_char,
            "end": ent.end_char
        })

    return {
        "entity_count": len(entities),
        "entities": entities
    }

# Membuat interface web panel sederhana dengan Gradio
demo = gr.Interface(
    fn=predict_entity,
    inputs=[
        gr.Textbox(label="Judul Artikel (Title)"),
        gr.Textbox(label="Isi Artikel (Body)", lines=5)
    ],
    outputs=gr.JSON(label="Hasil Ekstraksi Entitas"),
    title="Article Named-Entity Recognition Service",
    description="Masukkan judul dan isi artikel berita untuk mengekstrak entitasnya."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://23941c2a6237b98765.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
